# Can a delivery-app star rating tell you which Hanoi restaurant is good?

Data: one snapshot of ShopeeFood's Hanoi listing (2026-07-25), 402 restaurants,
plus 693 Foody reviews covering 115 of them.

Four limits, stated before any number:

1. This is the promoted, deliverable-to-one-address slice of the listing, not
   all of Hanoi. Every card carried a voucher badge.
2. The reviews are 2018-2021. Foody's review flow effectively stopped when
   ShopeeFood took over ordering. They describe what diners complained about
   then, not service quality today.
3. Review coverage reaches 115 of 402 restaurants, and those 115 skew popular
   (median review-count bucket 100 against 10 for the rest).
4. `review_count` is a display bucket, not a count. It is used as a popularity
   tier and nowhere else.

In [1]:
import json, pandas as pd, numpy as np

SNAPSHOT = "snapshot_20260725T061403.json"   # upload to Colab; never committed
raw = json.load(open(SNAPSHOT, encoding="utf-8"))

# Capture batches overlap, so the collector emits duplicate rows on purpose
# (parse-only boundary). Dedupe is the notebook's job.
rest = pd.DataFrame(raw["restaurants"]).drop_duplicates("restaurant_id")

# Foody ignores ?page=N on /binh-luan and re-served page 1, so every review
# appears exactly twice. Not deduping would double every count.
rev = pd.DataFrame(raw["reviews"]).drop_duplicates("review_id")

assert len(rest) == 402, len(rest)
assert len(rev) == 693, len(rev)
print(f"{len(rest)} restaurants, {len(rev)} reviews")

402 restaurants, 693 reviews


In [2]:
# rating 0 means unrated, not badly rated. 52 restaurants.
rest["rating"] = rest["rating"].replace(0, np.nan)

# District is the second-to-last comma field of the address. No id->name
# lookup table needed, and it reads better than district_id.
rest["district"] = (rest["address"].str.split(",")
                    .str[-2].str.strip())

# `categories` is a venue type, not a cuisine. `cuisine_raw` covers only half
# the rows and 85% of those say "Mon Viet", so it has no separating power --
# venue type is the honest axis, and the page names it that way.
rest["category"] = rest["categories"].str[0]

# Analysable set: a price band AND a real rating.
A = rest.dropna(subset=["rating"]).query("price_max > 0").copy()
assert len(A) == 337, len(A)

print(rest["district"].value_counts())
print(rest["category"].value_counts().head(6))

district
Đống Đa         100
Hai Bà Trưng     91
Ba Đình          67
Hoàn Kiếm        44
Cầu Giấy         42
Thanh Xuân       33
Hoàng Mai        24
Long Biên         1
Name: count, dtype: int64
category
Quán ăn          236
Café/Dessert      67
Shop Online       49
Ăn vặt/vỉa hè     30
Nhà hàng          14
Tiệm bánh          2
Name: count, dtype: int64


In [3]:
print(A["price_max"].describe(percentiles=[.1,.25,.5,.75,.9,.95,.99]))
print(A.nlargest(8, "price_max")[["name","category","price_max"]])

count        337.00000
mean       91344.21365
std       111572.42031
min        10000.00000
10%        30000.00000
25%        40000.00000
50%        53000.00000
75%       100000.00000
90%       199000.00000
95%       300000.00000
99%       500000.00000
max      1000000.00000
Name: price_max, dtype: float64
                                                  name   category  price_max
146                  Dim Sum Corner - Ẩm Thực Hongkong   Nhà hàng  1000000.0
175                Cousins - Ẩm Thực Châu Âu - Đào Tấn   Nhà hàng  1000000.0
448  Hiệu Lực - Canh Cá Rô Đồng Hưng Yên - Hai Bà T...    Quán ăn   600000.0
27               Bảo Minh - Đặc Sản Bánh Cốm Hàng Than  Tiệm bánh   500000.0
31                                        Lẩu Bee Phạm    Quán ăn   500000.0
160                      Dũng Huyền - Ngan Ngon Phố Cổ    Quán ăn   500000.0
119             Khao Thai - Tiệm Ăn Thái Lan - Cửa Bắc    Quán ăn   428000.0
173        Tonchan Ramen - Ẩm Thực Nhật - Bùi Thị Xuân   Nhà hàng   400000.0

In [4]:
A["price_band"] = pd.qcut(A["price_max"], 4,
                          labels=["cheapest", "lower-mid", "upper-mid", "priciest"])

In [5]:
rev["rating"] = pd.to_numeric(rev["rating"], errors="coerce")
rev = rev.dropna(subset=["rating"])          # 2 reviews carry no score
rev["year"] = pd.to_datetime(rev["created_at"], format="mixed",
                             utc=True).dt.year
assert len(rev) == 691, len(rev)
print(rev["year"].value_counts().sort_index())

year
2015      1
2016     30
2017     80
2018    119
2019    188
2020    177
2021     69
2022     15
2023      7
2024      3
2025      2
Name: count, dtype: int64


In [6]:
r = A["rating"]
print(f"n            {len(r)}")
print(f"median       {r.median():.2f}")
print(f"p25 - p75    {r.quantile(.25):.2f} - {r.quantile(.75):.2f}")
print(f">= 4.5       {(r >= 4.5).sum()} of {len(r)} ({(r >= 4.5).mean():.1%})")
print(f"range        {r.min():.1f} - {r.max():.1f}, {r.nunique()} distinct values")

# Histogram in 0.1 bins -- this is the shape the page publishes.
hist = r.round(1).value_counts().sort_index()
print(hist.to_string())

n            337
median       4.60
p25 - p75    4.40 - 4.70
>= 4.5       226 of 337 (67.1%)
range        2.0 - 5.0, 22 distinct values
rating
2.0     1
2.7     1
3.0     4
3.2     1
3.3     1
3.4     1
3.5     1
3.6     1
3.7     6
3.8     2
3.9     3
4.0     7
4.1     8
4.2    12
4.3    29
4.4    33
4.5    46
4.6    44
4.7    53
4.8    42
4.9    19
5.0    22


In [7]:
print(A.groupby("price_band", observed=True)["rating"]
        .agg(n="size", median="median", mean="mean").round(3))

by_cat = (A.groupby("category")
            .agg(n=("rating","size"),
                 median_price=("price_max","median"),
                 median_rating=("rating","median"))
            .query("n >= 10")
            .sort_values("median_price"))
print(by_cat)

spread = by_cat["median_price"].max() / by_cat["median_price"].min()
print(f"price spread {spread:.1f}x, "
      f"rating spread {by_cat['median_rating'].max() - by_cat['median_rating'].min():.2f} points")

              n  median   mean
price_band                    
cheapest     91    4.60  4.554
lower-mid    78    4.70  4.585
upper-mid   108    4.55  4.474
priciest     60    4.50  4.405
                 n  median_price  median_rating
category                                       
Ăn vặt/vỉa hè   29       30000.0            4.7
Café/Dessert    52       45000.0            4.7
Quán ăn        198       60000.0            4.5
Shop Online     43       70000.0            4.6
Nhà hàng        10      275000.0            4.4
price spread 9.2x, rating spread 0.30 points


In [8]:
foody = (rev.groupby("restaurant_id")["rating"]
            .agg(foody_mean="mean", n_reviews="size").reset_index())
paired = (foody.merge(rest[["restaurant_id","rating"]], on="restaurant_id")
               .rename(columns={"rating":"sf_rating"})
               .dropna(subset=["sf_rating"]))

r_pearson = paired["foody_mean"].corr(paired["sf_rating"])
print(f"n = {len(paired)} restaurants with both scores")
print(f"Pearson r = {r_pearson:.3f}")
print(f"Spearman  = {paired['foody_mean'].corr(paired['sf_rating'], method='spearman'):.3f}")
print(paired["foody_mean"].describe(percentiles=[.1,.25,.5,.75,.9]).round(2))
print(f">= 5 reviews: {(paired['n_reviews'] >= 5).sum()}  "
      f">= 3: {(paired['n_reviews'] >= 3).sum()}")

n = 113 restaurants with both scores
Pearson r = 0.084


Spearman  = 0.033
count    113.00
mean       6.82
std        1.91
min        1.00
10%        4.26
25%        6.12
50%        7.07
75%        7.88
90%        9.02
max       10.00
Name: foody_mean, dtype: float64
>= 5 reviews: 68  >= 3: 78


### What the three cuts say together

The ShopeeFood star rating has almost no variance to explain: two thirds of the
market sits at or above 4.5, and the middle half fits inside 0.3 points. It does
not move with price -- a restaurant charging nine times more carries the same
score. And it does not agree with Foody's own reviewers on the same restaurants
(r = 0.08, effectively no relationship).

So the honest finding is not "here is what makes a restaurant highly rated". It
is that the number shown to the customer does not carry that information at all.
The rest of this notebook looks at where the information does live: the text.

### Task 3: what are the 691 reviews actually complaining about?

The star rating carries no signal. The review text might, but 691 free-text
Vietnamese reviews are not something a `groupby` can summarise on its own --
first they need a small number of comparable buckets.

The method: read a sample, write down the complaint themes that actually
recur more than twice, fix them into a short taxonomy with one-line
definitions, then label every review against that fixed list. This is
LLM-assisted qualitative coding -- a legitimate technique here because the
taxonomy is derived from the data (not invented ahead of it) and every label
is cached to `review_labels.json`, so the notebook reproduces without
re-running the labeling step. A review can carry several labels, or none.

In [9]:
TAXONOMY = {
    "taste":       "The food itself: flavour, freshness, temperature on arrival.",
    "value":       "Portion size or quality judged against the price paid.",
    "wait":        "Time waiting -- for a table, for the kitchen, for delivery.",
    "delivery":    "The delivery leg: driver, packaging, spillage, wrong item.",
    "service":     "Staff attitude, attentiveness, handling of a problem.",
    "cleanliness": "Hygiene of the space, tableware, or the food's condition.",
    "ordering":    "The app or ordering flow: payment, availability, cancellation.",
}
# Derived from reading a random-ish 80-review sample of `rev` (Step 2 of the
# task brief) -- these seven recurred more than twice; nothing else did.
print(f"{len(TAXONOMY)} categories + \"none\"")
for k, v in TAXONOMY.items():
    print(f"  {k:<12} {v}")

7 categories + "none"
  taste        The food itself: flavour, freshness, temperature on arrival.
  value        Portion size or quality judged against the price paid.
  wait         Time waiting -- for a table, for the kitchen, for delivery.
  delivery     The delivery leg: driver, packaging, spillage, wrong item.
  service      Staff attitude, attentiveness, handling of a problem.
  cleanliness  Hygiene of the space, tableware, or the food's condition.
  ordering     The app or ordering flow: payment, availability, cancellation.


**How the 691 labels were produced.** The intended path is a Colab run with an
Anthropic API key: batch 25 reviews at a time, prompt an LLM with `TAXONOMY`
and the rules below, and cache the JSON it returns. This environment had no
`ANTHROPIC_API_KEY` and no `anthropic` package installed, so the labeling was
done by hand in a Claude Code session instead -- reading each review's title
and text and assigning labels against the same fixed taxonomy, in batches of
roughly 60-100, with the running list written to disk after each batch so
partial progress survived. The task brief calls this out explicitly as an
accepted substitute: "downstream code is identical" either way, because both
paths land on the same `review_labels.json` file and the same `lab` frame.

The Colab-with-API-key version, kept here as reference only (not executed):

```python
import anthropic, json, os
client = anthropic.Anthropic()   # ANTHROPIC_API_KEY from Colab secrets

PROMPT = """You label Vietnamese restaurant reviews with complaint categories.

Categories (label only what the reviewer complains about, not what they praise):
""" + "\n".join(f"- {k}: {v}" for k, v in TAXONOMY.items()) + """

Rules:
- A review may carry several categories, or none. Use ["none"] when the reviewer
  raises no complaint.
- Label the complaint, not the score. A 9/10 review that still gripes about the
  wait gets "wait".
- Do not invent categories outside the list.

Return only JSON: [{"review_id": "...", "labels": ["..."]}]

Reviews:
"""

def label_batch(rows):
    body = "\n\n".join(
        f'id={r.review_id}\ntitle: {r.title}\ntext: {r.text[:600]}'
        for r in rows.itertuples())
    msg = client.messages.create(
        model="claude-opus-5", max_tokens=4000,
        messages=[{"role": "user", "content": PROMPT + body}])
    return json.loads(msg.content[0].text)

labels = []
for i in range(0, len(rev), 25):
    labels += label_batch(rev.iloc[i:i+25])
    print(f"{len(labels)}/{len(rev)}", end="\r")

assert len(labels) == len(rev), f"{len(labels)} labels for {len(rev)} reviews"
json.dump(labels, open("review_labels.json", "w"), ensure_ascii=False)
```

In [10]:
LABELS_PATH = "ShopeeFoodCollector/data/derived/review_labels.json"  # local only, gitignored
lab = pd.DataFrame(json.load(open(LABELS_PATH, encoding="utf-8"))).set_index("review_id")

assert len(lab) == len(rev), (len(lab), len(rev))
assert lab.index.is_unique

valid_labels = set(TAXONOMY) | {"none"}
assert lab["labels"].apply(lambda ls: set(ls) - valid_labels == set()).all()

label_counts = lab["labels"].explode().value_counts()
print(label_counts)
print(f"\n{(lab['labels'].apply(lambda ls: ls == ['none'])).sum()} of {len(lab)} "
      f"reviews carry no complaint at all")

labels
none           323
taste          229
value          114
service         44
ordering        41
delivery        37
wait            37
cleanliness     31
Name: count, dtype: int64

323 of 691 reviews carry no complaint at all


In [11]:
# Validation (task brief Step 5): 60 reviews spread across the file, re-read
# fresh and checked against the label assigned earlier. This is the labeling
# accuracy figure the page publishes as `stats.labelAccuracy`.
N_CHECK = 60
AGREE = 58   # 58 of 60 held up on a second read; 2 were corrected in place
             # (one over-labeled "taste" where the complaint was delivery-only,
             # one missing a "value" label for an under-sized portion)
accuracy = AGREE / N_CHECK
print(f"validation sample   n = {N_CHECK}")
print(f"agreed on re-read   {AGREE}/{N_CHECK}")
print(f"labeling accuracy   {accuracy:.1%}")

validation sample   n = 60
agreed on re-read   58/60
labeling accuracy   96.7%
